In [65]:
import math

class Point:
    def __init__(self, x=0, y=0):
        self._x = x
        self._y = y

    def get_x(self):
        return self._x

    def set_x(self, x):
        self._x = x

    def get_y(self):
        return self._y

    def set_y(self, y):
        self._y = y

    def __eq__(self, other):
        if isinstance(other, Point):
            # Use math.isclose with a default relative tolerance of 1e-9 and an absolute tolerance of 0.0
            return math.isclose(self._x, other.get_x()) and math.isclose(self._y, other.get_y())
        return False

    def  __str__(self):
         return f'Point({self.get_x()}, {self.get_y()})'

    __repr__ = __str__

    def distance_to(self, other):
        return math.sqrt((self._x - other.get_x()) ** 2 + (self._y - other.get_y()) ** 2)

In [66]:
class Segment:
    def __init__(self, point1, point2):
        self._point1 = point1
        self._point2 = point2

    def  __str__(self):
         return f'Segment({self._point1}, {self._point2})'
        
    def _calculate_slope(self, p1, p2):
        if math.isclose(p1.get_x(), p2.get_x()):
            return float('inf')  # Vertical line
        return (p2.get_y() - p1.get_y()) / (p2.get_x() - p1.get_x())

    def _calculate_intercept(self, p, slope):
        if math.isinf(slope):  # Vertical line
            return p.get_x()
        return p.get_y() - slope * p.get_x()

    def is_parallel(self, other_segment):
        slope1 = self._calculate_slope(self._point1, self._point2)
        slope2 = self._calculate_slope(other_segment._point1, other_segment._point2)
        return math.isclose(slope1, slope2)

    def is_collinear(self, other_segment):
        if not self.is_parallel(other_segment):
            return False

        slope = self._calculate_slope(self._point1, self._point2)
        intercept1 = self._calculate_intercept(self._point1, slope)
        intercept2 = self._calculate_intercept(other_segment._point1, slope)
        return math.isclose(intercept1, intercept2)

    def is_point_on_segment(self, point):
        # Check if the point is within the x and y bounds of the segment
        x_min, x_max = sorted([self._point1.get_x(), self._point2.get_x()])
        y_min, y_max = sorted([self._point1.get_y(), self._point2.get_y()])
        within_x = x_min <= point.get_x() <= x_max
        within_y = y_min <= point.get_y() <= y_max

        # For vertical line, just check y bounds
        if math.isclose(self._point1.get_x(), self._point2.get_x()):
            return within_y

        # Calculate slope and intercept, then check if point satisfies line equation
        slope = self._calculate_slope(self._point1, self._point2)
        intercept = self._calculate_intercept(self._point1, slope)
        return within_x and within_y and math.isclose(point.get_y(), slope * point.get_x() + intercept)

    def get_intersection_point(self, other_segment):
        x1, y1 = self._point1.get_x(), self._point1.get_y()
        x2, y2 = self._point2.get_x(), self._point2.get_y()
        x3, y3 = other_segment._point1.get_x(), other_segment._point1.get_y()
        x4, y4 = other_segment._point2.get_x(), other_segment._point2.get_y()

        # For collinear case it is enough to check 3 points
        if self.is_collinear(other_segment):
            if self.is_point_on_segment(other_segment._point1):
                return other_segment._point1
            if self.is_point_on_segment(other_segment._point2):
                return other_segment._point2
            if other_segment.is_point_on_segment(self._point1):
                return self._point1
            return None
            
        # Calculate intersection point using line equations
        denominator = (x1 - x2) * (y3 - y4) - (y1 - y2) * (x3 - x4)
        if denominator == 0:  # Lines are parallel
            return None
        
        px = ((x1 * y2 - y1 * x2) * (x3 - x4) - (x1 - x2) * (x3 * y4 - y3 * x4)) / denominator
        py = ((x1 * y2 - y1 * x2) * (y3 - y4) - (y1 - y2) * (x3 * y4 - y3 * x4)) / denominator

        # Check if intersection point lies within both segments
        if (min(x1, x2) <= px <= max(x1, x2) and
            min(y1, y2) <= py <= max(y1, y2) and
            min(x3, x4) <= px <= max(x3, x4) and
            min(y3, y4) <= py <= max(y3, y4)):
            return Point(px, py)
        
        return None

In [67]:
class Triangle:
    def __init__(self, point1, point2, point3):
        self._point1 = point1
        self._point2 = point2
        self._point3 = point3
        self._segment1 = Segment(point1, point2)
        self._segment2 = Segment(point2, point3)
        self._segment3 = Segment(point1, point3)
        self._fermat_point = None

    def _sign(self, p1, p2, p3):
        return (p1.get_x() - p3.get_x()) * (p2.get_y() - p3.get_y()) - (p2.get_x() - p3.get_x()) * (p1.get_y() - p3.get_y())

    def is_point_inside(self, point):
        # Special case: all vertices are the same point
        if self._point1 == self._point2 and self._point2 == self._point3:
            return point == self._point1
        
        # Special case: triangle is a line (degenerate case)
        if self._segment1.is_collinear(self._segment2) and self._segment2.is_collinear(self._segment3):
            return self._segment1.is_point_on_segment(point) or self._segment2.is_point_on_segment(point) or self._segment3.is_point_on_segment(point)

        # Barycentric coordinate system
        d1 = self._sign(point, self._point1, self._point2)
        d2 = self._sign(point, self._point2, self._point3)
        d3 = self._sign(point, self._point3, self._point1)

        has_neg = (d1 < 0) or (d2 < 0) or (d3 < 0)
        has_pos = (d1 > 0) or (d2 > 0) or (d3 > 0)

        # The point is on the inside if there are no signs differing (all non-negative or all non-positive)
        return not(has_neg and has_pos)

    def compute_fermat_point(self):
        if self._fermat_point is not None:
            return self._fermat_point

        # Special case: Collinear or two vertices are the same
        if self._point1 == self._point2 or self._point1 == self._point3 or self._point2 == self._point3 or \
           self._segment1.is_collinear(self._segment2) or self._segment2.is_collinear(self._segment3) or self._segment1.is_collinear(self._segment3):
            # Find the vertex that lies between the other two vertices
            if self._segment1.is_point_on_segment(self._point3):
                self._fermat_point = self._point3
            elif self._segment2.is_point_on_segment(self._point1):
                self._fermat_point = self._point1
            else:
                self._fermat_point = self._point2
            return self._fermat_point
        # Check if any angle is 120 degrees or more
        if self._angle_is_large(self._point1, self._point2, self._point3) or \
           self._angle_is_large(self._point2, self._point3, self._point1) or \
           self._angle_is_large(self._point3, self._point1, self._point2):
            # The Fermat point is at the vertex with the large angle
            # Determine which vertex it is and set it as the Fermat point
            for p in [self._point1, self._point2, self._point3]:
                if self._angle_is_large(p, *[point for point in [self._point1, self._point2, self._point3] if point != p]):
                    self._fermat_point = p
                    return self._fermat_point

        # For triangles with all angles less than 120 degrees, construct equilateral triangles and find their intersections
        fp1 = self._construct_equilateral(self._point1, self._point2)
        fp2 = self._construct_equilateral(self._point2, self._point3)

        # Find intersection of lines (point1, fp1) and (point2, fp2)
        self._fermat_point = self._find_intersection(self._point1, fp1, self._point2, fp2)
        return self._fermat_point

    def _angle_is_large(self, p1, p2, p3):
        # Use the law of cosines to calculate the angle at p1 formed by p2 and p3
        a = math.dist([p2.get_x(), p2.get_y()], [p3.get_x(), p3.get_y()])
        b = math.dist([p1.get_x(), p1.get_y()], [p3.get_x(), p3.get_y()])
        c = math.dist([p1.get_x(), p1.get_y()], [p2.get_x(), p2.get_y()])
        # Ensure the value inside acos is within the range [-1, 1]
        cos_angle = max(min((b**2 + c**2 - a**2) / (2 * b * c), 1), -1)
        angle = math.acos(cos_angle)
        return angle >= (2 * math.pi / 3)  # Check if the angle is 120 degrees or more in radians

    def _construct_equilateral(self, p1, p2):
        # Construct an equilateral triangle on the side formed by p1 and p2
        angle = math.pi / 3  # 60 degrees in radians
        dx = p2.get_x() - p1.get_x()
        dy = p2.get_y() - p1.get_y()
        # Calculate the length of the side and the new direction
        length = math.sqrt(dx**2 + dy**2)
        new_dx = math.cos(angle) * dx - math.sin(angle) * dy
        new_dy = math.sin(angle) * dx + math.cos(angle) * dy
        # Return the third vertex of the equilateral triangle
        return Point(p1.get_x() + new_dx / length * math.sqrt(3) * length / 2,
                     p1.get_y() + new_dy / length * math.sqrt(3) * length / 2)

    def _find_intersection(self, p1, q1, p2, q2):
        # Calculate the intersection point of lines p1q1 and p2q2
        det = (q1.get_x() - p1.get_x()) * (q2.get_y() - p2.get_y()) - (q1.get_y() - p1.get_y()) * (q2.get_x() - p2.get_x())
        if det == 0:
            return None  # Lines are parallel, no intersection
        lambda_ = ((q2.get_y() - p2.get_y()) * (q2.get_x() - p1.get_x()) + (p2.get_x() - q2.get_x()) * (q2.get_y() - p1.get_y())) / det
        x = p1.get_x() + lambda_ * (q1.get_x() - p1.get_x())
        y = p1.get_y() + lambda_ * (q1.get_y() - p1.get_y())
        return Point(x, y)

In [68]:
class Obstacles:
    def __init__(self):
        self.triangles = []

    def add_triangle(self, triangle):
        self.triangles.append(triangle)

    def is_point_inside_any_obstacle(self, point):
        for triangle in self.triangles:
            if triangle.is_point_inside(point):
                return True  # The point is inside at least one triangle
        return False  # The point is not inside any triangle

    def __str__(self):
        return '\n'.join(str(triangle) for triangle in self.triangles)

    #checks if ends of segment have LOS connection
    def is_segment_clear(self, segment):
        for triangle in self.triangles:
            # Check if any endpoint of the segment is inside a triangle
            if triangle.is_point_inside(segment._point1) or triangle.is_point_inside(segment._point2):
                return False
            
            # Check for intersection between the segment and each side of the triangle
            for tri_segment in [Segment(triangle._point1, triangle._point2),
                                Segment(triangle._point2, triangle._point3),
                                Segment(triangle._point3, triangle._point1)]:
                if segment.get_intersection_point(tri_segment) is not None:
                    return False  # The segment intersects with a triangle side

        return True  # The segment is clear of all obstacles

In [69]:
'''

!!!COMPLEX PART STARTS HERE!!!

'''

'\n\n!!!COMPLEX PART STARTS HERE!!!\n\n'

In [76]:
# TODO: Works incorrectly, should be fixed

class ConnectivityComponent:
    def __init__(self):
        self.BSs = []
    def addBS(self, newBS):
        self.BSs.append(newBS)
    def __str__(self):
        # Generate a list of string representations of Point objects and join them with ', '
        bs_strings = [str(bs) for bs in self.BSs]
        component_str = "Connectivity Component: [" + ', '.join(bs_strings) + "]"
        return component_str
    __repr__ = __str__

# Responsible for initializing graph
class GraphBuilder:
    def __init__(self, radiusBS, radiusDroneBS, baseStations, powerStations=[], obstacles=[]):
        baseStations, powerStations = self._check_stations_over_obstacles(baseStations, powerStations, obstacles)
        self._components = self._find_components(baseStations, radiusBS, obstacles)
        self._powerStations = self._filter_covered_power_stations(powerStations, baseStations, radiusBS, obstacles)
        self._obstacles = obstacles
        self._radiusBS = radiusBS
        self._radiusDroneBS = radiusDroneBS

        print(self._components)

    def _check_stations_over_obstacles(self, baseStations, powerStations, obstacles):
        filtered_base_stations = []
        filtered_power_stations = []

        # Check base stations
        for base in baseStations:
            if not obstacles.is_point_inside_any_obstacle(base):
                filtered_base_stations.append(base)
            else:
                print(f"Base station at {base} is within an obstacle and will be filtered out.")

        # Check power stations
        for power in powerStations:
            if not obstacles.is_point_inside_any_obstacle(power):
                filtered_power_stations.append(power)
            else:
                print(f"Power station at {power} is within an obstacle and will be filtered out.")

        return filtered_base_stations, filtered_power_stations
    
    def _filter_covered_power_stations(self, powerStations, baseStations, radiusBS, obstacles):
        filtered_power_stations = []
        for power in powerStations:
            is_close_to_base_station = False
            for base in baseStations:
                if power.distance_to(base) <= radiusBS:
                    # Check if the segment between power and base is clear of obstacles
                    segment = Segment(power, base)
                    if obstacles.is_segment_clear(segment):
                        is_close_to_base_station = True
                        break  # No need to check further if one is found within radiusBS and clear of obstacles
            if not is_close_to_base_station:
                filtered_power_stations.append(power)
        return filtered_power_stations

    def _find_components(self, baseStations, radiusBS, obstacles):
        def dfs(node, visited, component):
            visited[node] = True
            component.addBS(baseStations[node])
            for neighbor in graph[node]:
                if not visited[neighbor]:
                    dfs(neighbor, visited, component)

        # Create the graph
        graph = {i: [] for i in range(len(baseStations))}
        for i, base1 in enumerate(baseStations):
            for j, base2 in enumerate(baseStations):
                if i != j and base1.distance_to(base2) <= radiusBS:
                    # Check if the segment between base1 and base2 is clear of obstacles
                    segment = Segment(base1, base2)
                    if obstacles.is_segment_clear(segment):
                        graph[i].append(j)

        # Find connected components using DFS
        visited = [False] * len(baseStations)
        components = []
        for i in range(len(baseStations)):
            if not visited[i]:
                component = ConnectivityComponent()
                dfs(i, visited, component)
                components.append(component)

        return components

In [77]:
radiusBS = 5    
radiusDroneBS = 3
baseStations = [Point(1, 1), Point(1, 0), Point(6, 0), Point(-10, 5), Point(-10, 6), Point(100, 100)]
powerStations = [Point(0, 0), Point(200, 200)]
obstacles = Obstacles()
obstacles.add_triangle(Triangle(Point(2, 0), Point(3, 0), Point(2.5, 10)))
graphBuilder = GraphBuilder(radiusBS, radiusDroneBS, baseStations, powerStations, obstacles)


[Connectivity Component: [Point(1, 1), Point(1, 0)], Connectivity Component: [Point(6, 0)], Connectivity Component: [Point(-10, 5), Point(-10, 6)], Connectivity Component: [Point(100, 100)]]


In [ ]:
# And implement the algorithm from the Lavesh's thesis

In [18]:
# I also have to implement metaheuristic to find a proper path